In [ ]:
!pip install transformers datasets pandas torch

In [ ]:
!wget https://www.cs.cmu.edu/~enron/enron_mail_20150507.tar.gz

--2026-03-14 10:59:14--  https://www.cs.cmu.edu/~enron/enron_mail_20150507.tar.gz
Resolving www.cs.cmu.edu (www.cs.cmu.edu)... 128.2.42.95
Connecting to www.cs.cmu.edu (www.cs.cmu.edu)|128.2.42.95|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 443254787 (423M) [application/x-gzip]
Saving to: ‘enron_mail_20150507.tar.gz’

enron_mail_20150507 100%[===================>] 422.72M  2.90MB/s    in 2m 49s  

2026-03-14 11:02:03 (2.50 MB/s) - ‘enron_mail_20150507.tar.gz’ saved [443254787/443254787]



In [ ]:
!tar -xzf enron_mail_20150507.tar.gz

In [ ]:
import os
print(os.listdir())

['.config', 'enron_mail_20150507.tar.gz', 'maildir', 'sample_data']


In [ ]:
os.listdir("maildir")[:10]

['kuykendall-t',
 'ermis-f',
 'hyvl-d',
 'cash-m',
 'scholtes-d',
 'crandell-s',
 'panus-s',
 'benson-r',
 'weldon-c',
 'mcconnell-m']

In [ ]:
import os
import pandas as pd

emails = []

for root, dirs, files in os.walk("maildir"):
    for file in files:
        path = os.path.join(root, file)

        try:
            with open(path, "r", errors="ignore") as f:
                text = f.read()

            subject = ""
            body = ""

            if "Subject:" in text:
                subject = text.split("Subject:")[1].split("\n")[0]

            if "\n\n" in text:
                body = text.split("\n\n",1)[1]

            if subject.strip() != "" and body.strip() != "":
                emails.append((body.strip(), subject.strip()))

        except:
            pass

df = pd.DataFrame(emails, columns=["email_body","subject"])

print("Dataset size:", len(df))
df.head()

Dataset size: 498164


,email_body,subject
0,hey there - i was just talking about you with ...,Re: Dinner Friday Night
1,we had a couple of different deal numbers that...,Re: Deal 528768 - December 2000
2,"not to be inconsiderate, but we did get him a ...",Re: Phillip
3,I'm so glad you wrote back. Isn't the interne...,Re:
4,Please click on the attached link to launch yo...,Holiday Invitation


In [ ]:
df = df.sample(5000, random_state=42)

print(len(df))
df.head()

5000


,email_body,subject
69318,Steve:\nI just wanted to thank you and Lindy f...,TW business plan
437117,I updated the NYPA volumes for November. The ...,NYPA for Nov
125559,thank you very much for th info.\n\n\n \n\tE...,Re: Floating LNG Terminal
342863,Just a status check....according to Melanie Ki...,DOE
455034,We have received the executed EEI Master Power...,Indianapolis Power and Light Company


In [ ]:
df = df.rename(columns={
    "email_body":"input_text",
    "subject":"target_text"
})

df["input_text"] = "generate subject: " + df["input_text"]

df.head()

,input_text,target_text
69318,generate subject: Steve:\nI just wanted to tha...,TW business plan
437117,generate subject: I updated the NYPA volumes f...,NYPA for Nov
125559,generate subject: thank you very much for th i...,Re: Floating LNG Terminal
342863,generate subject: Just a status check....accor...,DOE
455034,generate subject: We have received the execute...,Indianapolis Power and Light Company


In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

dataset

Dataset({
    features: ['input_text', 'target_text', '__index_level_0__'],
    num_rows: 5000
})

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

print("Model loaded")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Model loaded


In [ ]:
def tokenize(example):

    input_enc = tokenizer(
        example["input_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

    target_enc = tokenizer(
        example["target_text"],
        padding="max_length",
        truncation=True,
        max_length=32
    )

    input_enc["labels"] = target_enc["input_ids"]

    return input_enc


tokenized_dataset = dataset.map(tokenize)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="results",
    per_device_train_batch_size=4,
    num_train_epochs=1,
    logging_steps=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [ ]:
email = "Please attend the project meeting tomorrow at 10 AM"

input_text = "generate subject: " + email

inputs = tokenizer.encode(input_text, return_tensors="pt")

outputs = model.generate(inputs, max_length=20)

print("Generated Subject:", tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
email = "The finance report for Q3 is attached please review it before tomorrow"

input_text = "generate subject: " + email

inputs = tokenizer.encode(input_text, return_tensors="pt")

outputs = model.generate(inputs, max_length=20)

print("Generated Subject:", tokenizer.decode(outputs[0], skip_special_tokens=True))

Generated Subject: FQ Finance Report for Q3


In [ ]:
inputs = tokenizer.encode(input_text, return_tensors="pt")

outputs = model.generate(
    inputs,
    max_length=15,
    num_beams=5,
    no_repeat_ngram_size=2,
    early_stopping=True
)

print("Generated Subject:", tokenizer.decode(outputs[0], skip_special_tokens=True))

Generated Subject: generate subject subject: Team lunch scheduled on Friday
